# Explicativo del preprocesamiento y modelado con set de validación estático

- Correr los scripts `03_preprocessing.py`  y  `04a_pre_static_split.py`
- Ejecutar la notebook `05a_modeling_static.ipynb`

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, TimeSeriesSplit, cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

# =====================================================================
# 1. PREPARACIÓN Y LIMPIEZA (03_preprocessing.py)
# sobre train_before_eda.csv (85%) y test.csv (15%)
# salida: train_clean.csv (85%) y test_clean.csv (15%) ya con limpieza y feature engineering aplicados  
# =====================================================================

# =====================================================================
# 2. SPLITTING TEMPORAL [70% Train, 15% Validation, 15% Test] en (04a_pre_static_split.py)
# =====================================================================
# Primer split: Es lo primero se hizo, se hizo para el EDA. Luego pasó por la etapa de preparación y limpieza
# Segundo split: Del 85% restante, separamos un fragmento que equivalga al 15% del total original.
# 0.15 / 0.85 = 0.17647 (Fracción de df_temp destinada a Validation)
df_train, df_val = train_test_split(df_temp, test_size=(0.15/0.85), shuffle=False)

# Eliminamos fechas y target para crear las matrices X e y
drop_para_modelo = ['is_canceled', 'booking_date', 'arrival_date']

X_train = df_train.drop(columns=drop_para_modelo)
y_train = df_train['is_canceled']

X_val = df_val.drop(columns=drop_para_modelo)
y_val = df_val['is_canceled']

X_test = df_test.drop(columns=drop_para_modelo)
y_test = df_test['is_canceled']

# =====================================================================
# 3. PIPELINES DE PREPROCESAMIENTO Y MODELADO
# =====================================================================
vars_categoricas = [
    'meal', 'market_segment', 'distribution_channel', 
    'reserved_room_type', 'deposit_type', 'customer_type'
]

vars_numericas = [
    'lead_time', 'adr', 'total_nights', 'adults', 'children', 'babies', 
    'previous_cancellations', 'required_car_parking_spaces', 
    'total_of_special_requests', 'agent', 'company', 
    'arrival_date_year', 'arrival_month_num' 
]

vars_binarias = ['is_repeated_guest', 'is_placed_on_waiting_list', 'is_portugal', 'is_resort']

preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), vars_numericas),
        ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), vars_categoricas),
        ('bin', 'passthrough', vars_binarias)
    ], 
    remainder='drop'
)

# Definición de modelos
pipeline_lr = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', LogisticRegression(max_iter=1000, random_state=42, class_weight='balanced'))
])

pipeline_rf = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1, class_weight='balanced'))
])

# =====================================================================
# 4. ENTRENAMIENTO Y EXTRACCIÓN DE CARACTERÍSTICAS
# =====================================================================
pipeline_rf.fit(X_train, y_train)

# Extracción de nombres de variables para Feature Importance
preprocesador_entrenado = pipeline_rf.named_steps['preprocessor']
cat_encoder = preprocesador_entrenado.named_transformers_['cat']
cat_features = cat_encoder.get_feature_names_out(vars_categoricas).tolist()

all_features = vars_numericas + cat_features + vars_binarias
print(f"Número total de variables procesadas: {len(all_features)}")

Los algoritmos de Machine Learning tradicionales en scikit-learn, como la Regresión Logística o los Random Forest, son estrictamente operaciones matemáticas. Estos modelos solo entienden números (enteros o decimales) y no tienen la capacidad nativa de procesar, multiplicar o dividir objetos de tipo texto o fecha (datetime) como '2015-07-01'.

Si intentamos pasar las columnas booking_date o arrival_date directamente a la fase de entrenamiento, el compilador arrojará un error técnico (TypeError o ValueError). El modelo no sabe cómo interpretar una fecha cruda.

Por eso las eliminamos del espacio de características ($X$), pero no descartamos la información que contienen. Antes de tirarlas, aplicamos Ingeniería de Características (Feature Engineering) para extraer su valor analítico y transformarlo en variables numéricas o categóricas que el algoritmo sí pueda procesar. De esas fechas ya hemos extraído:

- Estacionalidad: Usamos el mes numérico (arrival_month_num) para que el modelo aprenda patrones estacionales (por ejemplo, si en verano hay más cancelaciones).
- Comportamiento del usuario: Calculamos el lead_time (que es la diferencia matemática entre la fecha de reserva y la de llegada) para medir la anticipación en días.
- Tendencias a largo plazo: Conservamos el año (arrival_date_year) para entender si el volumen de cancelaciones cambia anualmente.

Existe también una razón fundamental de sobreajuste (overfitting). Si forzamos a un modelo a aprender usando fechas exactas convirtiéndolas en números absolutos (timestamps), el algoritmo podría memorizar que el "12 de octubre de 2016" hubo muchas cancelaciones. Esa regla específica no le servirá de nada para predecir el comportamiento de un cliente en octubre de 2027. Al extraer el mes, el año y los días de anticipación, le enseñamos al modelo a generalizar patrones de comportamiento en lugar de memorizar un calendario que no se volverá a repetir.

El parámetro `class_weight='balanced'` interviene directamente en las matemáticas internas del algoritmo durante el entrenamiento para evitar ese comportamiento perezoso.

- Penalización asimétrica: Normalmente, un falso positivo (predecir que cancelará cuando no lo hace) le cuesta al modelo exactamente lo mismo que un falso negativo (predecir que no cancelará cuando en realidad sí lo hace). Al activar este parámetro, alteramos esa regla.

- Cálculo de pesos automático: Scikit-learn asigna un peso multiplicador a cada clase de forma inversamente proporcional a su frecuencia en los datos. Al ser minoritaria la clase "Cancelado", los errores que cometa el modelo al intentar predecirla se multiplicarán y dolerán mucho más en su función de pérdida.

- Cambio de enfoque: Obliga al algoritmo a dejar de buscar la precisión global fácil y lo fuerza a esforzarse por igual en aprender los patrones de ambas clases, dándole artificialmente la misma importancia matemática al 37% de cancelaciones que al 63% de estadías efectivas.